# Squares in Squares - validate and score the best packings
**Score: 22.13520** (lower is better). The packings come from the dataset *Squares in Squares - Best Packings*. Its base is the shortest-per-n merge of the newest public outputs by **Penguin**, **Nilanjan Saha**, **Sung Heo (kmmmvs)**, **Areeb Ali**, **jazivxt** (Boxed In Sub), **Malyshev Danil** (Living Atlas) and others, which in turn merge the work of lucifer19, Alex Chilton, avazaku and many more, built on Evan Daniel's and Francisco Couzo's MIT tools and the Soft-to-Rigid engine by Yohei Nakajima. Our own contribution is small: puzzles shortened by basin hopping and certified with the competition metric (n=585 is still ours; a new round is running).

> **Copy & Edit this notebook, set `WRITE_SUBMISSION = True`, Run All**, then submit the written `submission.csv`.

In [ ]:
import math, glob, os
import numpy as np, pandas as pd
from numba import njit

WRITE_SUBMISSION = True   # True -> writes submission.csv to the output folder

cand = glob.glob('/kaggle/input/**/submission.csv', recursive=True) or glob.glob('submission.csv')
SRC = cand[0]; print('using', SRC)
sub = pd.read_csv(SRC, dtype=str)
print(sub.shape, list(sub.columns))

In [ ]:
@njit(cache=True)
def _overlap(x, y, a, i, j, eps):
    dx = x[j]-x[i]; dy = y[j]-y[i]
    if dx*dx + dy*dy > 2.0: return False
    ci, si, cj, sj = math.cos(a[i]), math.sin(a[i]), math.cos(a[j]), math.sin(a[j])
    for k in range(4):
        if k == 0: ax, ay = ci, si
        elif k == 1: ax, ay = -si, ci
        elif k == 2: ax, ay = cj, sj
        else: ax, ay = -sj, cj
        ri = 0.5*(abs(ax*ci+ay*si) + abs(-ax*si+ay*ci))
        rj = 0.5*(abs(ax*cj+ay*sj) + abs(-ax*sj+ay*cj))
        if ri + rj - abs(dx*ax+dy*ay) <= eps: return False   # separating axis found
    return True

@njit(cache=True)
def count_overlaps(x, y, a, eps):
    order = np.argsort(x); c = 0
    for ii in range(len(x)):
        i = order[ii]
        for jj in range(ii+1, len(x)):
            j = order[jj]
            if x[j]-x[i] > 1.5: break
            if _overlap(x, y, a, i, j, eps): c += 1
    return c

def side(x, y, a):
    h = 0.5*(np.abs(np.cos(a)) + np.abs(np.sin(a)))
    return max((x+h).max()-(x-h).min(), (y+h).max()-(y-h).min())

In [ ]:
rows = []; total = 0.0; bad = []
for n, g in sub.groupby('n', sort=True):
    n = int(n); g = g.sort_values('square_id')
    x = pd.to_numeric(g.x).to_numpy(np.float64); y = pd.to_numeric(g.y).to_numpy(np.float64); a = pd.to_numeric(g.angle).to_numpy(np.float64)
    ok = len(x) == n and np.isfinite(x).all() and np.isfinite(y).all() and np.isfinite(a).all()
    ov = count_overlaps(x, y, a, 1e-12) if ok else -1
    if not ok or ov: bad.append((n, ok, ov))
    L = side(x, y, a); s = L/math.sqrt(n) - 1; total += s
    rows.append((n, L, s, ov))
res = pd.DataFrame(rows, columns=['n', 'L', 'score_n', 'overlaps'])
print(f'total score = {total:.12f}   puzzles = {len(res)}   flagged = {len(bad)}')
print('NOTE: float64 check only; the official metric certifies with interval arithmetic.')

In [ ]:
res['band_k'] = np.ceil(np.sqrt(res.n)).astype(int)
band = res.groupby('band_k').score_n.agg(['sum', 'mean', 'max']).round(4)
display(band.head(12)); print('bands 1..32 sum =', round(band['sum'].sum(), 4))

In [ ]:
if WRITE_SUBMISSION:
    sub.to_csv('submission.csv', index=False); print('written submission.csv')
else:
    print('WRITE_SUBMISSION is False: no submission file is written.')